In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from statsmodels.tsa.ar_model import AutoReg
import matplotlib.pyplot as plt
import copy
import json
import os
import random
import time
import zlib
import warnings
from joblib import Parallel, delayed
from pathlib import Path
warnings.filterwarnings('ignore')

# Random seeds
np.random.seed(42)
torch.manual_seed(42)

In [3]:
# LSTM Config
class Config_LSTM:
    use_aic: bool = True
    data_source = Path('..') / 'Data'
    data_files = [
        'AAPL.csv',
        'AMGN.csv',
        'AMZN.csv',
        'AXP.csv',
        'BA.csv',
        'CAT.csv',
        'CSCO.csv',
        'CVX.csv',
        'DIS.csv',
        'GS.csv',
        'HD.csv',
        'HON.csv',
        'IBM.csv',
        'JNJ.csv',
        'JPM.csv',
        'KO.csv',
        'MCD.csv',
        'MMM.csv',
        'MRK.csv',
        'MSFT.csv',
        'NKE.csv',
        'NVDA.csv',
        'PG.csv',
        'SHW.csv',
        'TRV.csv',
        'UNH.csv',
        'WMT.csv'
    ]
    estimate_type = 'QMLE-Trade'
    value_column = 'Volatility'
    sample_start_date = '2000-01-01'
    sample_end_date   = '2026-06-30'
    output_root = Path('..') / 'Results'

    # A symbol-date with more than one observation of the selected estimate
    # type: 'drop' treats the date as missing (default); 'first' keeps the
    # first source row.
    duplicate_policy = 'drop'

    # Memory. lookback_period is used only when use_aic is False. The AIC
    # search ceiling equals the longest fixed lookback in the grid
    # {10, 21, 42, 63}, so AIC never selects a longer lookback than the
    # longest fixed one.
    lookback_period     = 42
    max_aic_lag         = 63
    # AIC selection sample. None = all observations from sample_start up to
    # the selection date (main specification). An integer = rolling window of
    # that many years (not used in the paper; own output folder).
    aic_selection_years = None
    # How often the lookback is re-selected, in months, counted from the first
    # test window (1 January 2004). 12 = each January on all data before that
    # January (main specification; with test_months = 12 this coincides with
    # every refit). 1 = re-selected at every monthly refit (own output folder).
    aic_reselect_months = 12
    # Placebo: path to a CSV with columns selection_date, lookback giving one
    # lookback per re-selection date, common to all stocks (the cross-stock
    # median of the AIC lookback). None = off. Output folder: LSTM_PLACEBO.
    lookback_schedule   = None
    # Added to every window seed. 0 = main runs; 1 = an independent seed set
    # for the noise-floor run (output folder gets a _seed1 suffix).
    seed_offset         = 0

    # Walk-forward calendar. test_months is the refit cadence:
    # 12 = yearly (main specification), 1 = monthly (robustness run).
    initial_train_years = 3
    validation_years    = 1
    test_months         = 12

    hidden_layers         = [16, 8]
    learning_rate         = 0.001
    batch_size            = 64
    epochs                = 100
    early_stopping_rounds = 10
    n_ensembles           = 10
    random_seed            = 42
    batch_normalization    = False

    # Portable CPU multiprocessing defaults. Override cpu_workers after
    # constructing the config if a particular machine needs a different value.
    cpu_workers             = 24
    torch_threads_per_worker = 1
    parallel_verbose         = 0

    use_log_transform = True
    target_scaler     = True

    device = torch.device('cpu')

    @property
    def lag_candidates(self):
        """Fixed-lookback lags, oldest first: lag_L, ..., lag_1.

        Derived from lookback_period on every access, so setting
        config.lookback_period on an instance also changes the inputs.
        """
        return list(range(int(self.lookback_period), 0, -1))


print(f"Using CPU with {Config_LSTM.cpu_workers} parallel window workers")

Using CPU with 24 parallel window workers


In [4]:
class LSTMForecaster(nn.Module):
    def __init__(self, input_dim: int, hidden_layers, dropout: float = 0.0):
        super().__init__()

        self.input_dim = int(input_dim)
        self.hidden_layers = list(hidden_layers)
        self.dropout = float(dropout)

        # Build LSTM stack
        self.lstm_layers = nn.ModuleList()
        self.lstm_layers.append(nn.LSTM(
            input_size=1,
            hidden_size=self.hidden_layers[0],
            num_layers=1,
            batch_first=True
        ))

        for in_size, out_size in zip(self.hidden_layers[:-1], self.hidden_layers[1:]):
            self.lstm_layers.append(nn.LSTM(
                input_size=in_size,
                hidden_size=out_size,
                num_layers=1,
                batch_first=True
            ))

        self.use_dropout = self.dropout > 0.0
        if self.use_dropout:
            self.dropout_layer = nn.Dropout(self.dropout)

        last_hidden = self.hidden_layers[-1]
        self.fc_out = nn.Linear(last_hidden, 1)

        nn.init.kaiming_uniform_(self.fc_out.weight, a=np.sqrt(5))
        if self.fc_out.bias is not None:
            fan_in, _ = nn.init._calculate_fan_in_and_fan_out(self.fc_out.weight)
            bound = 1 / np.sqrt(fan_in)
            nn.init.uniform_(self.fc_out.bias, -bound, bound)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(-1)
        elif x.dim() == 3:
            pass
        else:
            raise ValueError(f"Expected input of shape (N, L) or (N, L, C), got {tuple(x.shape)}")

        for lstm in self.lstm_layers:
            x, _ = lstm(x)

        last = x[:, -1, :]
        if self.use_dropout:
            last = self.dropout_layer(last)

        out = self.fc_out(last)
        return out

In [5]:
# Data loading and panel construction
def parse_source_dates(values):
    """Parse ISO dates and day-first slash dates without ambiguity."""
    text = values.astype(str).str.strip()
    slash_mask = text.str.contains('/', regex=False)
    parsed = pd.Series(pd.NaT, index=text.index, dtype='datetime64[ns]')
    parsed.loc[~slash_mask] = pd.to_datetime(
        text.loc[~slash_mask], format='%Y-%m-%d', errors='raise'
    )
    parsed.loc[slash_mask] = pd.to_datetime(
        text.loc[slash_mask], dayfirst=True, errors='raise'
    )
    return parsed


def resolve_duplicate_observations(frame, policy, estimate_type, keys=('Date', 'Symbol')):
    """Handle symbol-dates with more than one observation.

    policy='drop'  : the symbol-date is treated as missing (all its rows removed)
    policy='first' : the first source row is retained
    """
    keys = list(keys)
    duplicate_rows = frame.duplicated(keys, keep=False)
    if not duplicate_rows.any():
        return frame
    duplicate_keys = frame.loc[duplicate_rows, keys].drop_duplicates()
    if policy == 'drop':
        frame = frame.loc[~duplicate_rows]
        action = 'treated as missing'
    elif policy == 'first':
        frame = frame.drop_duplicates(keys, keep='first')
        action = 'first source observation retained'
    else:
        raise ValueError("duplicate_policy must be 'drop' or 'first'")
    print(
        f'{len(duplicate_keys)} symbol-date(s) had repeated {estimate_type} '
        f'observations; {action}:'
    )
    for _, row in duplicate_keys.iterrows():
        print(f"  {row['Symbol']} {pd.Timestamp(row['Date']).date()}")
    return frame


def load_dacheng_xiu_panel(
    data_directory,
    data_files,
    estimate_type='QMLE-Trade',
    value_column='Volatility',
    sample_start_date='2000-01-01',
    sample_end_date='2026-06-30',
    duplicate_policy='drop'
):
    data_directory = Path(data_directory)
    if not data_directory.is_dir():
        raise FileNotFoundError(f"Data directory does not exist: {data_directory}")
    if not data_files:
        raise ValueError("Add at least one Dacheng Xiu CSV filename to data_files")

    csv_paths = []
    for filename in data_files:
        filename_path = Path(filename)
        if filename_path.suffix == '':
            filename_path = filename_path.with_suffix('.csv')
        csv_path = data_directory / filename_path
        if not csv_path.is_file():
            raise FileNotFoundError(f"Selected CSV does not exist: {csv_path}")
        csv_paths.append(csv_path)

    if len(set(csv_paths)) != len(csv_paths):
        raise ValueError("data_files contains duplicate filenames")

    sample_start = pd.Timestamp(sample_start_date)
    sample_end = pd.Timestamp(sample_end_date)
    if sample_start > sample_end:
        raise ValueError('sample_start_date must not be after sample_end_date')

    required_columns = {'Symbol', 'PN', 'Type', 'Date', value_column}
    selected_frames = []

    for csv_path in csv_paths:
        source_data = pd.read_csv(csv_path)
        missing_columns = required_columns.difference(source_data.columns)
        if missing_columns:
            raise ValueError(
                f"{csv_path.name} is missing columns: {sorted(missing_columns)}"
            )

        ticker_data = source_data.loc[
            source_data['Type'].eq(estimate_type),
            ['Symbol', 'Date', value_column]
        ].copy()
        if ticker_data.empty:
            raise ValueError(
                f"{csv_path.name} contains no rows for estimate type '{estimate_type}'"
            )

        ticker_data['Date'] = parse_source_dates(ticker_data['Date'])
        ticker_data[value_column] = pd.to_numeric(
            ticker_data[value_column], errors='raise'
        )
        # Retain pre-sample observations as lag history, but never retain
        # observations after the configured sample end.
        ticker_data = ticker_data.loc[
            ticker_data['Date'] <= sample_end
        ].copy()
        target_sample = ticker_data.loc[
            ticker_data['Date'].between(sample_start, sample_end, inclusive='both')
        ]
        if target_sample.empty:
            raise ValueError(
                f'{csv_path.name} has no {estimate_type} observations between '
                f'{sample_start.date()} and {sample_end.date()}'
            )
        selected_frames.append(ticker_data)

    combined_data = pd.concat(selected_frames, ignore_index=True)
    combined_data = resolve_duplicate_observations(
        combined_data, duplicate_policy, estimate_type
    )

    panel = combined_data.pivot(
        index='Date', columns='Symbol', values=value_column
    ).sort_index()
    panel.index.name = 'date'
    panel.columns.name = None

    print(
        f"Constructed panel from {len(csv_paths)} Dacheng Xiu CSV file(s) "
        f"using {estimate_type}; {value_column} values were not squared"
    )
    print(f"Panel shape: {panel.shape[0]} dates x {panel.shape[1]} symbols")
    print(
        f"Loaded history: {panel.index.min().date()} to {panel.index.max().date()} "
        f"| target sample: {sample_start.date()} to {sample_end.date()}"
    )
    return panel


def get_stock_series(rv_df, ticker):
    s = rv_df[ticker]
    # Identify the stock's own start/end (first/last non-NaN)
    valid = s.dropna()
    if len(valid) == 0:
        raise ValueError(f"No data for ticker {ticker}")
    stock_start = valid.index[0]
    stock_end   = valid.index[-1]
    print(f"  [{ticker}] Data: {stock_start.date()} to {stock_end.date()} ({len(valid)} obs)")
    return s  # keep NaNs for global alignment

def create_lagged_features(series, lags, use_log_transform=True):
    df = pd.DataFrame({'RV_daily': series})
    df['log_RV'] = np.log(df['RV_daily'].clip(lower=1e-8))
    feature_source = df['log_RV'] if use_log_transform else df['RV_daily']
    for lag in lags:
        df[f'lag_{lag}'] = feature_source.shift(lag)
    df = df.dropna()
    return df

In [6]:
# AIC lag selection
def lookback_selection_date(test_start, config):
    """Date up to which the lookback is selected for a test window.

    Re-selection happens every aic_reselect_months months counted from the
    first test window, so with 12 the lookback is chosen each January on all
    data before that January and held fixed for the monthly refits in
    between. With 1 it is the test start itself.
    """
    test_start = pd.Timestamp(test_start)
    first_test = (
        pd.Timestamp(config.sample_start_date)
        + pd.DateOffset(years=config.initial_train_years)
        + pd.DateOffset(years=config.validation_years)
    )
    step = max(1, int(config.aic_reselect_months))
    months = (
        (test_start.year - first_test.year) * 12
        + (test_start.month - first_test.month)
    )
    return first_test + pd.DateOffset(months=(months // step) * step)


def aic_selection_start_date(selection_date, config):
    """First date of the AIC selection sample for a selection date."""
    sample_start = pd.Timestamp(config.sample_start_date)
    if config.aic_selection_years is None:
        return sample_start
    return max(
        sample_start,
        pd.Timestamp(selection_date) - pd.DateOffset(years=int(config.aic_selection_years)),
    )


def load_lookback_schedule(path):
    """Placebo schedule: {selection_date: lookback}, common to all stocks."""
    schedule = pd.read_csv(path, parse_dates=['selection_date'])
    return dict(zip(schedule['selection_date'], schedule['lookback'].astype(int)))


def calculate_AIC(selection_data, max_lag: int = 63, verbose: bool = True):
    """AR(p) by conditional ML (OLS) with an intercept, p = 1..max_lag.

    hold_back=max_lag fits every order on the same observations, so the
    AIC values are comparable across p.
    """
    aic_values = {}
    for lag in range(1, max_lag + 1):
        model  = AutoReg(
            selection_data, lags=lag, hold_back=max_lag
        )
        result = model.fit()
        aic_values[lag] = result.aic

    best_lag = min(aic_values, key=aic_values.get)

    lookback = list(range(best_lag, 0, -1))
    if verbose:
        print(f"    AIC-selected lookback period: {best_lag} ({len(lookback)} lags)")
    return lookback, best_lag

In [7]:
# Training utilities
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0):
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best_loss  = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter   = 0

def train_single_model(X_train, y_train, X_val, y_val, config, verbose=False):
    X_train_tensor = torch.FloatTensor(X_train).to(config.device)
    y_train_tensor = torch.FloatTensor(y_train).to(config.device)
    X_val_tensor   = torch.FloatTensor(X_val).to(config.device)
    y_val_tensor   = torch.FloatTensor(y_val).to(config.device)

    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    train_loader  = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)

    model     = LSTMForecaster(X_train.shape[1], config.hidden_layers).to(config.device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)

    early_stopping = EarlyStopping(patience=config.early_stopping_rounds)
    best_val   = float('inf')
    best_state = None
    best_epoch = 0

    for epoch in range(config.epochs):
        model.train()
        epoch_train_loss = 0.0
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            out  = model(batch_X)
            loss = criterion(out, batch_y)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_train_loss += loss.item()

        avg_train_loss = epoch_train_loss / max(1, len(train_loader))

        model.eval()
        with torch.inference_mode():
            vout  = model(X_val_tensor)
            vloss = criterion(vout, y_val_tensor).item()

        if (best_val - vloss) > early_stopping.min_delta:
            best_val   = vloss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1

        early_stopping(vloss)
        if early_stopping.early_stop:
            if verbose:
                print(f"Early stopping at epoch {epoch+1}; best={best_epoch} val={best_val:.6f}")
            break

        if verbose and (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{config.epochs}  Train: {avg_train_loss:.6f}  Val: {vloss:.6f}")

    if best_state is not None:
        model.load_state_dict(best_state)
    # Diagnostics recorded in the per-window log.
    model.best_epoch = best_epoch
    model.epochs_run = epoch + 1
    return model

def set_seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def train_ensemble(
    X_train, y_train, X_val, y_val, config, base_seed=42, verbose=True
):
    models    = []
    if verbose:
        print(f"    Training ensemble of {config.n_ensembles} models...")
    for i in range(config.n_ensembles):
        set_seed_all(base_seed + i)
        model = train_single_model(X_train, y_train, X_val, y_val, config, verbose=False)
        models.append(model)
    return models

def predict_ensemble(models, X, config):
    X_tensor    = torch.FloatTensor(X).to(config.device)
    predictions = []
    for model in models:
        model.eval()
        with torch.no_grad():
            predictions.append(model(X_tensor).cpu().numpy())
    return np.stack(predictions, axis=0)

def ensemble_forecast_levels(models, X_scaled, target_scaler, config):
    """Ensemble forecast in RV levels.

    Each member's forecast is inverse-scaled to log RV and exponentiated;
    the ensemble forecast is the mean of the members' level forecasts.
    Used for both test and validation forecasts so they are combined
    identically.
    """
    pred_scaled_all = predict_ensemble(models, X_scaled, config)
    if config.target_scaler and target_scaler is not None:
        log_all = np.stack(
            [target_scaler.inverse_transform(p) for p in pred_scaled_all], axis=0
        )
    else:
        log_all = pred_scaled_all
    if config.use_log_transform:
        level = np.mean(np.exp(log_all), axis=0)
    else:
        level = np.mean(log_all, axis=0)
    return np.maximum(level, 1e-12).flatten()

In [8]:
# Metrics
def calculate_mse(y_true_vol, y_pred_vol):
    """MSE of variance forecasts, computed from volatility-level inputs.

    On variances, MSE is robust to noise in the realised measure
    (Patton, 2011); on volatilities it is not.
    """
    y_true_var = np.square(np.asarray(y_true_vol, dtype=float))
    y_pred_var = np.square(np.asarray(y_pred_vol, dtype=float))
    return mean_squared_error(y_true_var, y_pred_var)

def calculate_qlike(y_true_vol, y_pred_vol):
    """Variance QLIKE computed from volatility-level observations and forecasts."""
    y_true_var = np.square(np.maximum(np.asarray(y_true_vol, dtype=float), 1e-8))
    y_pred_var = np.square(np.maximum(np.asarray(y_pred_vol, dtype=float), 1e-8))
    ratio = y_true_var / y_pred_var
    return np.mean(ratio - np.log(ratio) - 1)

In [9]:
# Per-stock expanding window forecast
def _expanding_window_forecast_single_sequential(
    ticker: str,
    rv_series: pd.Series,   # raw RV values
    global_index: pd.DatetimeIndex,
    config,
    only_test_start=None,
    window_seed=None,
    window_index=None,
    verbose=True
):
    """Returns (predictions, selected lookbacks, per-window records).

    Each record holds the window's dates, the number of lag inputs used,
    the ensemble's validation-year QLIKE/MSE (used to build LSTM_VAL),
    timing, and early-stopping diagnostics.
    """
    stock_rv = rv_series.dropna()          # restrict to the stock's own date range

    use_schedule = config.lookback_schedule is not None
    if use_schedule:
        schedule = load_lookback_schedule(config.lookback_schedule)
    if config.use_aic or use_schedule:
        lags = list(range(1, config.max_aic_lag + 1))
    else:
        lags = config.lag_candidates
        feature_cols = [f'lag_{lag}' for lag in config.lag_candidates]
    df_with_lags = create_lagged_features(
        stock_rv, lags, use_log_transform=config.use_log_transform
    )

    # Lag construction uses the available pre-sample history. Target rows and
    # all split boundaries are anchored to the common configured sample dates,
    # so the lookback length cannot shift the evaluation calendar.
    sample_start = pd.Timestamp(config.sample_start_date)
    sample_end = pd.Timestamp(config.sample_end_date)
    df_with_lags = df_with_lags.loc[
        (df_with_lags.index >= sample_start)
        & (df_with_lags.index <= sample_end)
    ].copy()
    current_test_start = (
        sample_start
        + pd.DateOffset(years=config.initial_train_years)
        + pd.DateOffset(years=config.validation_years)
    )
    if only_test_start is not None:
        current_test_start = pd.Timestamp(only_test_start)

    pred_dict = {}       # date -> predicted RV
    aic_dict  = {}       # date -> chosen lookback period (int)
    window_records = []  # one dict per re-estimation window

    window_count = 0

    while current_test_start <= sample_end:
        window_count += 1

        validation_start = (
            current_test_start - pd.DateOffset(years=config.validation_years)
        )
        next_test_start = current_test_start + pd.DateOffset(months=config.test_months)

        # Half-open training, validation, and test windows prevent overlap.
        # Pre-sample observations may appear in lag columns, but never as targets.
        # training   = [sample_start, validation_start)
        # validation = [validation_start, current_test_start)
        # test       = [current_test_start, next_test_start)
        train_data = df_with_lags.loc[
            (df_with_lags.index >= sample_start)
            & (df_with_lags.index < validation_start)
        ]
        val_data = df_with_lags.loc[
            (df_with_lags.index >= validation_start)
            & (df_with_lags.index < current_test_start)
        ]
        test_data = df_with_lags.loc[
            (df_with_lags.index >= current_test_start)
            & (df_with_lags.index < next_test_start)
        ]

        # Memory selection sample: every observation from the sample start up
        # to the selection date, i.e. all data available when the lookback is
        # (re-)selected. With aic_reselect_months = 12 the selection date is
        # the most recent 1 January, so the lookback is held fixed for the
        # monthly refits within a year.
        selection_date = lookback_selection_date(current_test_start, config)
        aic_selection_start = aic_selection_start_date(selection_date, config)
        aic_selection_data = df_with_lags.loc[
            (df_with_lags.index >= aic_selection_start)
            & (df_with_lags.index < selection_date)
        ].copy()

        if (len(test_data) == 0 or len(train_data) == 0 or len(val_data) == 0
                or (config.use_aic and len(aic_selection_data) == 0)):
            break

        if verbose:
            print(f"  [{ticker}] Window {window_count}: "
                  f"train {train_data.index[0].date()}–{train_data.index[-1].date()} "
                  f"| val {val_data.index[0].date()}–{val_data.index[-1].date()} "
                  f"| test {test_data.index[0].date()}–{test_data.index[-1].date()} "
                  f"({len(test_data)} obs)")

        # lag selection
        aic_seconds = np.nan
        if config.use_aic:
            aic_started = time.perf_counter()
            aic_lags, best_lag_int = calculate_AIC(
                aic_selection_data['log_RV'], config.max_aic_lag, verbose=verbose
            )
            aic_seconds = time.perf_counter() - aic_started
            current_feature_cols   = [f'lag_{lag}' for lag in aic_lags]
        elif use_schedule:
            if selection_date not in schedule:
                raise KeyError(
                    f'No placebo lookback for selection date {selection_date.date()}'
                )
            best_lag_int = int(schedule[selection_date])
            current_feature_cols = [f'lag_{lag}' for lag in range(best_lag_int, 0, -1)]
        else:
            current_feature_cols = feature_cols
            best_lag_int         = None

        # features & targets
        X_train = train_data[current_feature_cols].values
        X_val   = val_data[current_feature_cols].values
        X_test  = test_data[current_feature_cols].values

        y_train_original = train_data['RV_daily'].values.reshape(-1, 1)
        y_val_original   = val_data['RV_daily'].values.reshape(-1, 1)

        if config.use_log_transform:
            y_train = np.log(np.maximum(y_train_original, 1e-8))
            y_val   = np.log(np.maximum(y_val_original,   1e-8))
        else:
            y_train = y_train_original.copy()
            y_val   = y_val_original.copy()

        # scaling (fitted on the training window only)
        feature_scaler  = StandardScaler()
        X_train_scaled  = feature_scaler.fit_transform(X_train)
        X_val_scaled    = feature_scaler.transform(X_val)
        X_test_scaled   = feature_scaler.transform(X_test)

        if config.target_scaler:
            target_scaler   = StandardScaler()
            y_train_scaled  = target_scaler.fit_transform(y_train)
            y_val_scaled    = target_scaler.transform(y_val)
        else:
            y_train_scaled  = y_train
            y_val_scaled    = y_val
            target_scaler   = None

        # train & predict
        train_started = time.perf_counter()
        models = train_ensemble(
            X_train_scaled,
            y_train_scaled,
            X_val_scaled,
            y_val_scaled,
            config,
            base_seed=config.random_seed if window_seed is None else window_seed,
            verbose=verbose
        )
        train_seconds = time.perf_counter() - train_started

        test_pred = ensemble_forecast_levels(models, X_test_scaled, target_scaler, config)

        # Validation-year forecast of the trained ensemble, combined exactly
        # like the test forecast. Its QLIKE is the selection criterion for
        # LSTM_VAL (lowest validation QLIKE across the fixed-lookback grid).
        val_pred = ensemble_forecast_levels(models, X_val_scaled, target_scaler, config)
        y_val_level = y_val_original.flatten()

        # store predictions aligned to global index
        for i, date in enumerate(test_data.index):
            pred_dict[date] = test_pred[i]
            if best_lag_int is not None:
                aic_dict[date] = best_lag_int   # same lookback for the whole test window

        window_records.append({
            'ticker': ticker,
            'window': window_count if window_index is None else int(window_index),
            'test_start': current_test_start.date().isoformat(),
            'test_end_exclusive': next_test_start.date().isoformat(),
            'validation_start': validation_start.date().isoformat(),
            'train_start': train_data.index[0].date().isoformat(),
            'n_train': int(len(train_data)),
            'n_validation': int(len(val_data)),
            'n_test': int(len(test_data)),
            'lookback': int(len(current_feature_cols)),
            'selection_date': selection_date.date().isoformat(),
            'val_qlike': float(calculate_qlike(y_val_level, val_pred)),
            'val_mse': float(calculate_mse(y_val_level, val_pred)),
            # Validation-year log forecast errors of the ensemble. Their variance
            # gives the lognormal bias correction exp(var/2) using only data
            # available at the forecast origin; not applied to the forecasts.
            'val_log_resid_mean': float(np.mean(
                np.log(np.maximum(y_val_level, 1e-8)) - np.log(val_pred))),
            'val_log_resid_var': float(np.var(
                np.log(np.maximum(y_val_level, 1e-8)) - np.log(val_pred), ddof=1)),
            'aic_seconds': float(aic_seconds),
            'train_seconds': float(train_seconds),
            'mean_best_epoch': float(np.mean([m.best_epoch for m in models])),
            'mean_epochs_run': float(np.mean([m.epochs_run for m in models])),
        })

        current_test_start = next_test_start
        if only_test_start is not None:
            break

    # reindex to global calendar
    pred_series = pd.Series(pred_dict, name=ticker, dtype=float).reindex(global_index)
    aic_series  = pd.Series(aic_dict,  name=ticker, dtype=float).reindex(global_index)

    return pred_series, aic_series, window_records

In [10]:
# Portable CPU parallelism: one complete ensemble per window job.
def format_duration(seconds):
    total_seconds = max(0, int(seconds))
    hours, remainder = divmod(total_seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


def forecast_window_job(
    ticker,
    window_count,
    test_start,
    rv_series,
    global_index,
    config
):
    warnings.filterwarnings('ignore')
    torch.set_num_threads(config.torch_threads_per_worker)

    # Stable across worker counts and scheduling order.
    ticker_seed = zlib.crc32(ticker.encode('utf-8'))
    seed_limit = 2**31 - 1
    # seed_offset shifts every seed far beyond the range used by the main
    # runs (window_count * n_ensembles stays below 2,000), giving an
    # independent seed set for the noise-floor run.
    window_seed = (
        config.random_seed
        + int(config.seed_offset) * 100_000
        + ticker_seed
        + window_count * config.n_ensembles
    ) % seed_limit

    pred_s, aic_s, records = _expanding_window_forecast_single_sequential(
        ticker=ticker,
        rv_series=rv_series,
        global_index=global_index,
        config=config,
        only_test_start=test_start,
        window_seed=window_seed,
        window_index=window_count,
        verbose=False
    )
    return window_count, pred_s.dropna(), aic_s.dropna(), records


def expanding_window_forecast_single(
    ticker: str,
    rv_series: pd.Series,
    global_index: pd.DatetimeIndex,
    config
):
    sample_start = pd.Timestamp(config.sample_start_date)
    sample_end = pd.Timestamp(config.sample_end_date)
    test_start = (
        sample_start
        + pd.DateOffset(years=config.initial_train_years)
        + pd.DateOffset(years=config.validation_years)
    )
    window_starts = []
    while test_start <= sample_end:
        window_starts.append(test_start)
        test_start += pd.DateOffset(months=config.test_months)

    print(
        f"  [{ticker}] Dispatching {len(window_starts)} windows to "
        f"{config.cpu_workers} CPU workers"
    )
    results = Parallel(
        n_jobs=config.cpu_workers,
        backend='loky',
        batch_size=1,
        return_as='generator_unordered',
        verbose=config.parallel_verbose
    )(
        delayed(forecast_window_job)(
            ticker,
            window_count,
            current_test_start,
            rv_series,
            global_index,
            config
        )
        for window_count, current_test_start
        in enumerate(window_starts, start=1)
    )

    pred_dict = {}
    aic_dict = {}
    window_records = []
    completed_windows = 0
    total_windows = len(window_starts)
    started_at = time.perf_counter()
    for window_count, pred_s, aic_s, records in results:
        completed_windows += 1
        elapsed = time.perf_counter() - started_at
        eta = (elapsed / completed_windows) * (total_windows - completed_windows)
        window_records.extend(records)

        if pred_s.empty:
            print(
                f"  [{ticker}] {completed_windows}/{total_windows} complete | "
                f"window {window_count} produced no predictions | "
                f"elapsed {format_duration(elapsed)} | ETA {format_duration(eta)}",
                flush=True
            )
            continue
        pred_dict.update(pred_s.to_dict())
        aic_dict.update(aic_s.to_dict())

        actual_s = rv_series.reindex(pred_s.index)
        metric_mask = np.isfinite(actual_s.values) & np.isfinite(pred_s.values)
        if metric_mask.any():
            actual_values = actual_s.values[metric_mask]
            predicted_values = pred_s.values[metric_mask]
            window_mse = calculate_mse(actual_values, predicted_values)
            window_qlike = calculate_qlike(actual_values, predicted_values)
            metric_status = (
                f" | MSE {window_mse:.6g} | QLIKE {window_qlike:.6g}"
            )
        else:
            metric_status = " | MSE n/a | QLIKE n/a"
        if records:
            metric_status += f" | val QLIKE {records[-1]['val_qlike']:.6g}"

        aic_status = ""
        if config.use_aic and not aic_s.empty:
            aic_status = f" | AIC lag {int(aic_s.iloc[0])}"
        print(
            f"  [{ticker}] {completed_windows}/{total_windows} complete | "
            f"window {window_count} | "
            f"test {pred_s.index[0].date()}-{pred_s.index[-1].date()} "
            f"({len(pred_s)} obs){aic_status}{metric_status} | "
            f"elapsed {format_duration(elapsed)} | ETA {format_duration(eta)}",
            flush=True
        )

    print(
        f"  [{ticker}] Completed {completed_windows}/{total_windows} windows "
        f"in {format_duration(time.perf_counter() - started_at)}"
    )
    pred_series = pd.Series(pred_dict, name=ticker, dtype=float).reindex(global_index)
    aic_series = pd.Series(aic_dict, name=ticker, dtype=float).reindex(global_index)
    window_records = sorted(window_records, key=lambda record: record['window'])
    return pred_series, aic_series, window_records

In [11]:
# Structured output helpers and all-stock runner

def get_lstm_output_layout(config):
    if config.lookback_schedule is not None:
        mode_tag = 'PLACEBO'
    elif config.use_aic:
        mode_tag = 'AIC'
    else:
        mode_tag = str(config.lookback_period)
    model_name = f'LSTM_{mode_tag}'
    if config.use_aic and config.aic_selection_years is not None:
        # Rolling-window AIC selection, e.g. LSTM_AIC_aic3y.
        model_name += f'_aic{int(config.aic_selection_years)}y'
    if (config.use_aic or config.lookback_schedule is not None) and int(config.aic_reselect_months) != 12:
        # Non-default re-selection cadence, e.g. LSTM_AIC_resel1m.
        model_name += f'_resel{int(config.aic_reselect_months)}m'
    if int(config.seed_offset) != 0:
        # Noise-floor run with an independent seed set, e.g. LSTM_21_seed1.
        model_name += f'_seed{int(config.seed_offset)}'
    if int(config.test_months) != 12:
        # Robustness runs get their own folder, e.g. LSTM_AIC_refit1m.
        model_name += f'_refit{int(config.test_months)}m'
    output_dir = Path(config.output_root) / model_name
    individual_dir = output_dir / 'individual'
    return model_name, output_dir, individual_dir


def save_individual_stock_output(
    ticker,
    pred_s,
    aic_s,
    actual_s,
    individual_dir,
    config
):
    individual_dir.mkdir(parents=True, exist_ok=True)
    detail_df = pd.concat(
        [actual_s.rename('real'), pred_s.rename('predicted')], axis=1
    )
    valid_mask = (
        np.isfinite(detail_df['real'].to_numpy())
        & np.isfinite(detail_df['predicted'].to_numpy())
    )
    detail_df = detail_df.loc[valid_mask].copy()
    detail_df.insert(0, 'ticker', ticker)
    # Squared error of the variance forecast (MSE is reported on variances).
    detail_df['mse'] = (
        np.square(detail_df['real']) - np.square(detail_df['predicted'])
    ) ** 2
    actual_variance = np.square(detail_df['real'].clip(lower=1e-8))
    predicted_variance = np.square(detail_df['predicted'].clip(lower=1e-8))
    ratio = actual_variance / predicted_variance
    detail_df['qlike'] = ratio - np.log(ratio) - 1.0

    if config.use_aic or config.lookback_schedule is not None:
        detail_df['lookback_period'] = pd.to_numeric(
            aic_s.reindex(detail_df.index), errors='coerce'
        ).astype('Int64')
    else:
        detail_df['lookback_period'] = int(config.lookback_period)

    detail_df.index.name = 'date'
    detail_df = detail_df.reset_index()
    detail_columns = ['ticker', 'date', 'real', 'predicted', 'mse', 'qlike', 'lookback_period']
    detail_df = detail_df[detail_columns]
    safe_ticker = ''.join(
        char if char.isalnum() or char in '-_.' else '_'
        for char in str(ticker)
    )
    individual_path = individual_dir / f'{safe_ticker}.csv'
    detail_df.to_csv(individual_path, index=False)
    return individual_path


def run_all_stocks(rv_df: pd.DataFrame, config, individual_dir) -> tuple:
    sample_start = pd.Timestamp(config.sample_start_date)
    sample_end = pd.Timestamp(config.sample_end_date)
    global_index = rv_df.index[
        (rv_df.index >= sample_start) & (rv_df.index <= sample_end)
    ]
    tickers      = rv_df.columns.tolist()

    pred_cols = {}
    aic_cols  = {}
    all_records = []

    for ticker in tickers:
        print(f"\n{'='*60}")
        print(f"Processing: {ticker}")
        print(f"{'='*60}")
        try:
            pred_s, aic_s, records = expanding_window_forecast_single(
                ticker      = ticker,
                rv_series   = rv_df[ticker],
                global_index= global_index,
                config      = config
            )
            pred_cols[ticker] = pred_s
            aic_cols[ticker]  = aic_s
            all_records.extend(records)
            individual_path = save_individual_stock_output(
                ticker=ticker,
                pred_s=pred_s,
                aic_s=aic_s,
                actual_s=rv_df[ticker].reindex(global_index),
                individual_dir=individual_dir,
                config=config
            )
            print(f"  [{ticker}] Saved individual output: {individual_path}")
        except Exception as e:
            print(f"  [{ticker}] FAILED: {e}")
            pred_cols[ticker] = pd.Series(np.nan, index=global_index, name=ticker)
            aic_cols[ticker]  = pd.Series(np.nan, index=global_index, name=ticker)

    pred_df = pd.DataFrame(pred_cols, index=global_index)
    aic_df  = pd.DataFrame(aic_cols,  index=global_index)
    windows_df = pd.DataFrame(all_records)

    return pred_df, aic_df, windows_df

In [12]:
# Main entry point

def build_overall_evaluation(pred_df, input_df, model_name):
    actual_df = input_df.reindex(index=pred_df.index, columns=pred_df.columns)
    actual_values = actual_df.to_numpy(dtype=float)
    predicted_values = pred_df.to_numpy(dtype=float)
    valid_mask = np.isfinite(actual_values) & np.isfinite(predicted_values)

    stock_mse = []
    stock_qlike = []
    for column_index in range(valid_mask.shape[1]):
        stock_mask = valid_mask[:, column_index]
        if not stock_mask.any():
            continue
        stock_actual = actual_values[stock_mask, column_index]
        stock_predicted = predicted_values[stock_mask, column_index]
        stock_mse.append(calculate_mse(stock_actual, stock_predicted))
        stock_qlike.append(calculate_qlike(stock_actual, stock_predicted))

    overall_mse = np.mean(stock_mse) if stock_mse else np.nan
    overall_qlike = np.mean(stock_qlike) if stock_qlike else np.nan
    evaluated_tickers = len(stock_mse)
    return pd.DataFrame([{
        'model': model_name,
        'requested_tickers': int(pred_df.shape[1]),
        'evaluated_tickers': int(evaluated_tickers),
        'observations': int(valid_mask.sum()),
        'mse': overall_mse,
        'qlike': overall_qlike,
    }])


def check_lookbacks(windows_df, config):
    """Fail loudly if a fixed-lookback run did not use L inputs."""
    if windows_df.empty or config.use_aic or config.lookback_schedule is not None:
        return
    used = set(windows_df['lookback'].unique().tolist())
    if used != {int(config.lookback_period)}:
        raise RuntimeError(
            f'Fixed-lookback run expected {config.lookback_period} lag inputs '
            f'but windows used {sorted(used)}'
        )


def save_lstm_outputs(pred_df, aic_df, windows_df, input_df, config):
    model_name, output_dir, individual_dir = get_lstm_output_layout(config)
    output_dir.mkdir(parents=True, exist_ok=True)
    check_lookbacks(windows_df, config)

    summary_path = output_dir / f'{model_name}.csv'
    pred_df.to_csv(summary_path, index_label='date')

    evaluation_df = build_overall_evaluation(pred_df, input_df, model_name)
    evaluation_path = output_dir / f'{model_name}_evaluation.csv'
    evaluation_df.to_csv(evaluation_path, index=False)

    # One row per stock and re-estimation window: inputs used, validation
    # loss (for LSTM_VAL), timing and early-stopping diagnostics.
    windows_path = output_dir / f'{model_name}_windows.csv'
    windows_df.to_csv(windows_path, index=False)

    config_path = output_dir / f'{model_name}_config.json'
    output_paths = {
        'prediction_summary': str(summary_path),
        'evaluation': str(evaluation_path),
        'window_log': str(windows_path),
        'individual_directory': str(individual_dir),
        'run_config': str(config_path),
    }

    def window_stat(column, how):
        if windows_df.empty or column not in windows_df:
            return None
        value = getattr(windows_df[column], how)()
        return None if pd.isna(value) else float(value)

    non_nan_predictions = int(pred_df.notna().sum().sum())
    run_config = {
        'model': model_name,
        'generated_at_utc': pd.Timestamp.now(tz='UTC').isoformat(),
        'use_aic': bool(config.use_aic),
        'lookback_schedule': (
            None if config.lookback_schedule is None else str(config.lookback_schedule)
        ),
        'fixed_lookback_period': (
            None if (config.use_aic or config.lookback_schedule is not None)
            else int(config.lookback_period)
        ),
        'aic_reselect_months': int(config.aic_reselect_months),
        'seed_offset': int(config.seed_offset),
        'mse_definition': 'mean squared error of variance forecasts',
        'lag_inputs_used_min': window_stat('lookback', 'min'),
        'lag_inputs_used_max': window_stat('lookback', 'max'),
        'max_aic_lag': int(config.max_aic_lag),
        'aic_selection_window': (
            'expanding (sample start to selection date)'
            if config.aic_selection_years is None
            else f'rolling {int(config.aic_selection_years)} years'
        ),
        'initial_train_years': int(config.initial_train_years),
        'validation_years': int(config.validation_years),
        'test_months': int(config.test_months),
        'duplicate_policy': config.duplicate_policy,
        'hidden_layers': list(config.hidden_layers),
        'learning_rate': float(config.learning_rate),
        'batch_size': int(config.batch_size),
        'epochs': int(config.epochs),
        'early_stopping_rounds': int(config.early_stopping_rounds),
        'n_ensembles': int(config.n_ensembles),
        'random_seed': int(config.random_seed),
        'batch_normalization': bool(config.batch_normalization),
        'parallel_strategy': 'cpu_windows',
        'cpu_workers': int(config.cpu_workers),
        'torch_threads_per_worker': int(config.torch_threads_per_worker),
        'use_log_transform': bool(config.use_log_transform),
        'target_scaler': bool(config.target_scaler),
        'estimate_type': config.estimate_type,
        'value_column': config.value_column,
        'requested_sample_start_date': config.sample_start_date,
        'requested_sample_end_date': config.sample_end_date,
        'data_source': str(config.data_source),
        'data_files': list(config.data_files),
        'load_all_csv_files': not bool(config.data_files),
        'history_start_date': input_df.index.min().isoformat(),
        'history_end_date': input_df.index.max().isoformat(),
        'history_dates': int(input_df.shape[0]),
        'target_sample_start_date': config.sample_start_date,
        'target_sample_end_date': config.sample_end_date,
        'target_sample_dates': int(
            ((input_df.index >= pd.Timestamp(config.sample_start_date))
             & (input_df.index <= pd.Timestamp(config.sample_end_date))).sum()
        ),
        'symbols': input_df.columns.tolist(),
        'symbol_count': int(input_df.shape[1]),
        'prediction_shape': list(pred_df.shape),
        'non_nan_predictions': non_nan_predictions,
        'windows_logged': int(len(windows_df)),
        'mean_train_seconds_per_window': window_stat('train_seconds', 'mean'),
        'mean_aic_seconds_per_window': window_stat('aic_seconds', 'mean'),
        'device': str(config.device),
        'output_files': output_paths,
    }

    with config_path.open('w', encoding='utf-8') as config_file:
        json.dump(run_config, config_file, indent=2)

    return output_dir, output_paths

def main_lstm(config=None):
    print("="*60)
    print("LSTM NEURAL NETWORK – ALL STOCKS")
    print("="*60)

    config = Config_LSTM() if config is None else config
    if config.use_aic and config.lookback_schedule is not None:
        raise ValueError('Set either use_aic or lookback_schedule, not both')
    print(f"\nUsing device: {config.device}")
    print(f"AIC enabled:  {config.use_aic}")
    model_name, output_dir, individual_dir = get_lstm_output_layout(config)
    individual_dir.mkdir(parents=True, exist_ok=True)
    print(f"Output model: {model_name}")

    # Load data
    print(f"\nLoading data from: {config.data_source}")
    rv_df = load_dacheng_xiu_panel(
        config.data_source,
        config.data_files,
        estimate_type=config.estimate_type,
        value_column=config.value_column,
        sample_start_date=config.sample_start_date,
        sample_end_date=config.sample_end_date,
        duplicate_policy=config.duplicate_policy
    )

    # Run forecasts
    pred_df, aic_df, windows_df = run_all_stocks(rv_df, config, individual_dir)

    # Save output
    output_dir, output_paths = save_lstm_outputs(pred_df, aic_df, windows_df, rv_df, config)
    print(f"\nSaved model outputs to: {output_dir}")
    for output_name, output_path in output_paths.items():
        print(f"  {output_name}: {output_path}")

    print(f"\nOutput shape: {pred_df.shape[0]} dates x {pred_df.shape[1]} stocks")
    non_nan = pred_df.notna().sum().sum()
    print(f"Total predictions made: {non_nan:,}")

    return pred_df, aic_df, windows_df

# Runs are started from the run_variant cell below. (A bare main_lstm() call
# here would start an LSTM_42 run whenever this cell is executed.)

In [14]:
# Run helper: one call per model; each writes its own folder under ../Results.
def run_variant(**overrides):
    """Run main_lstm with Config_LSTM settings overridden for this run only."""
    config = Config_LSTM()
    for key, value in overrides.items():
        if not hasattr(config, key):
            raise AttributeError(f'Config_LSTM has no setting {key!r}')
        setattr(config, key, value)
    return main_lstm(config)

# Main results: uncomment these five; they run back to back, each writing
# its own folder under ../Results. Set cpu_workers to your number of cores.
# run_variant(lookback_period=10, cpu_workers=8)                    # LSTM_10
# run_variant(lookback_period=21, cpu_workers=8)                    # LSTM_21
# run_variant(lookback_period=42, cpu_workers=8)                    # LSTM_42
# run_variant(lookback_period=63, cpu_workers=8)                    # LSTM_63
run_variant(use_aic=True, cpu_workers=24)                          # LSTM_AIC
#
# Optional, not in the main results (leave commented for now):
# run_variant(lookback_schedule='lookback_schedule_placebo.csv')    # LSTM_PLACEBO (common lookback)
# run_variant(lookback_period=21, seed_offset=1)                    # LSTM_21_seed1 (noise floor)
# Monthly refits (12x the training): add test_months=1 to a call, e.g.
# run_variant(use_aic=True, test_months=1)                          # LSTM_AIC_refit1m


LSTM NEURAL NETWORK – ALL STOCKS

Using device: cpu
AIC enabled:  True
Output model: LSTM_AIC

Loading data from: ..\Data
3 symbol-date(s) had repeated QMLE-Trade observations; treated as missing:
  JNJ 2023-08-21
  JNJ 2023-08-22
  JNJ 2023-08-23
Constructed panel from 27 Dacheng Xiu CSV file(s) using QMLE-Trade; Volatility values were not squared
Panel shape: 7650 dates x 27 symbols
Loaded history: 1996-01-02 to 2026-06-30 | target sample: 2000-01-01 to 2026-06-30

Processing: AAPL
  [AAPL] Dispatching 23 windows to 24 CPU workers
  [AAPL] 1/23 complete | window 1 | test 2004-01-02-2004-12-31 (252 obs) | AIC lag 9 | MSE 0.00372761 | QLIKE 0.13541 | val QLIKE 0.093716 | elapsed 00:00:25 | ETA 00:09:20
  [AAPL] 2/23 complete | window 2 | test 2005-01-03-2005-12-30 (252 obs) | AIC lag 31 | MSE 0.00336142 | QLIKE 0.117178 | val QLIKE 0.111712 | elapsed 00:00:38 | ETA 00:06:42
  [AAPL] 3/23 complete | window 5 | test 2008-01-02-2008-12-31 (253 obs) | AIC lag 20 | MSE 0.113804 | QLIKE 0.17

(                AAPL      AMGN      AMZN       AXP        BA       CAT  \
 date                                                                     
 2000-01-03       NaN       NaN       NaN       NaN       NaN       NaN   
 2000-01-04       NaN       NaN       NaN       NaN       NaN       NaN   
 2000-01-05       NaN       NaN       NaN       NaN       NaN       NaN   
 2000-01-06       NaN       NaN       NaN       NaN       NaN       NaN   
 2000-01-07       NaN       NaN       NaN       NaN       NaN       NaN   
 ...              ...       ...       ...       ...       ...       ...   
 2026-06-24  0.245804  0.207090  0.262744  0.188256  0.258000  0.318828   
 2026-06-25  0.220192  0.219071  0.275154  0.184335       NaN  0.312130   
 2026-06-26  0.310723  0.225377  0.301836  0.205779  0.259108  0.330876   
 2026-06-29  0.341027  0.224522  0.333896  0.216938  0.251415  0.347868   
 2026-06-30  0.278869  0.241216  0.323575  0.201392  0.251250  0.331899   
 
                 CSCO  